# Suite XF — Transforms: models, procedures, application and certificates

A transform has a prototype, a procedure and a certificate ([CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md)). Its
model is mbse-schemas data, installed by `Transforms.install` and defined by `Transforms.define`: its name, its
prototype (a pattern, which declares its parameters through `Parametrics`) and the name its procedure is registered
under. A procedure is a native function registered by name, called with the caller's visitors and the parameter
assignments, and that's that: `procedure(source, sync, target, assignments)`, each a `Visitors.OfPattern` or None.
`Transforms.apply` receives the caller's visitors, the transform and the assignments, and calls the procedure, and
that's that; the caller verifies, if at all, with its own call; then `Transforms.certify`, given the same, the
certificates' store, the history and who decided the parameters, writes the certificates ([CAP-075: steps linked to what they matched and wrote](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-075.md)), in the one transaction mbse-transforms opens: a history certificate,
the sequence of transform certificates and who decided each; a transform certificate per application, with its
parameter assignments; and a sub-certificate per argument present, its pattern instance and the transform before it
on the same history.

The tests play the caller: each makes its visitors within its own transactions, from frames it obtained from its
stores (`support.bear`, which makes a `support.Frame`, the caller's `Visitors.OfPattern`), and passes them in. The
fixture is a shelf: a singleton `shelf.Shelf` owning `shelf.Item`s by key, each with a `name` and a `label`; the
pattern `shelf.Unlabelled`, an item without a label, which declares the parameter `case`; and `shelf.Whole`, the
shelf. The transforms are defined in a store of their own, `transforms`, and the certificates written in another,
`certs`.

In [ ]:
from contextlib import ExitStack

from mbse.Schemas.Framework import Builders as B, Parametrics, Plain, Proxies, References
from mbse.Transforms import Transforms
from support import bear, raises


def shelf(txn):
    """shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name and a label."""
    S = txn.Schemas
    return txn.uid("xf.shelf/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Shelf": S.OfObject, "Item": S.OfObject, "Holds": S.OfRelation})
        .spec([
            lambda _: _.Shelf.name("shelf.Shelf").ref(True).singleton("shelf.TheShelf").adjacencies({
                "items": lambda t: t.relation([_.Holds]).me("shelf")}),
            lambda _: _.Item.name("shelf.Item").ref(True).properties({"name": S.OfStr, "label": S.OfStr}).adjacencies([
                lambda t: t.name("shelf").relation([_.Holds]).me("item").owner(True)]),
            lambda _: _.Holds.name("shelf.Holds").links([lambda t: t.name("shelf"), lambda t: t.name("item")])
            .properties({"key": S.OfStr}).uniques([["shelf", "key"], ["item"]]),
        ])]))])


def patterns(txn):
    """shelf.Unlabelled, an item without a label, declaring the parameter case; and shelf.Whole, the shelf."""
    S = txn.Schemas
    return txn.uid("xf.patterns/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Unlabelled": S.OfPattern, "Whole": S.OfPattern})
        .spec([
            lambda _: _.Unlabelled.name("shelf.Unlabelled").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.label(None)])
            .parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "case": lambda d: d.parameter(lambda p: p.description("upper or lower").for_value([S.OfStr]))}))]),
            lambda _: _.Whole.name("shelf.Whole").symbols({"s": txn.shelf.Shelf}),
        ])]))])


def model(*names):
    """A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it."""
    store = B.Store()
    store.transaction(Parametrics.install)
    store.transaction(shelf)
    store.transaction(patterns)
    store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"s": t.shelf.Shelf})]))]))
    for name in names:
        with store.transaction():
            store.shelf.Shelf(store.singleton("shelf.TheShelf"), lambda s, n=name: s.items({
                n: lambda e: e.item(lambda i: i.name(n))}))
    return store


def unlabelled(store, name):
    """The frame of shelf.Unlabelled binding the item `name`, as `store.transaction` gives it back."""
    (frame,) = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
        lambda p: p.symbols({"i": t.shelf.Item}).spec([lambda _: _.i.name(name).label(None)])]))]))
    return frame


def whole(store):
    """The frame of shelf.Whole, as `store.transaction` gives it back."""
    held = store.registered("shelf.Whole")
    (frame,) = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([held]))]))
    return frame


def items(store):
    """Each item on the shelf: its key, name and label."""
    with store.transaction():
        return [(e.key.get(), e.item.name.get(), e.item.label.get()) for e in store.singleton("shelf.TheShelf").items]


def assignments_of(certificate):
    """The parameter assignments a transform certificate records, read through the visitor protocol."""
    held = {}

    def each(entries):
        for name, value in entries:
            value.as_scalar(lambda s, name=name: held.__setitem__(name, s.get()) or s)
        return entries

    certificate.assignments.as_value(lambda a: a.as_container(lambda c: c.as_entries(each) or c) or a)
    return held


def plain(store, value):
    """A value object a certificate holds, as plain data."""
    held = []
    value.as_object(lambda o: held.append(Plain.ToPlain(store)(o.as_schema(), o)) or o)
    return held[0]


def steps(history):
    """The number of transform certificates in a history."""
    with certs.transaction():
        try:
            return len(certs.singleton("Transforms.TheHistories").histories.at(history).history.steps)
        except KeyError:
            return 0


transforms = model()  # the transforms' store: the shelf's schemas and patterns, no data, and the transforms
transforms.transaction(Transforms.install)
for define in (
    lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled),
    lambda t: Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled),
):
    transforms.transaction(define)
certs = B.Store()  # the certificates' store, as the caller gives it
certs.transaction(Transforms.install)
procedures = Transforms.Procedures()


def application(name, assignments, history, by, **arguments):
    """The caller's application: within a transaction of the transforms' store and of each argument's store, the
    transform's visitor, and each argument's visitor, made from the frame given, `(store, pattern, frame)`, applied and
    then certified; the certificate's reference."""
    with ExitStack() as transactions:
        for store in {id(store): store for store in [transforms, *(a[0] for a in arguments.values())]}.values():
            transactions.enter_context(store.transaction())
        transform = transforms.singleton("Transforms.TheCatalog").transforms.at(name).transform
        visitors = {argument: bear(*given) for argument, given in arguments.items()}
        Transforms.apply(transform, assignments, procedures, **visitors)
        return Transforms.certify(certs, transform, assignments, history, by, **visitors)


def label(source, sync, target, assignments):
    """Labels the sync's item with its name, in the case its parameter assignment gives."""
    item = sync.i
    name = str(item.name)
    Proxies.store_of(item).shelf.Item(item, lambda i: i.label(name.upper() if assignments["case"] == "upper" else name.lower()))


procedures.register("label", label)


def copy(source, sync, target, assignments):
    """Puts on the target's shelf an item named as the source's, in the case its parameter assignment gives."""
    name = str(source.i.name)
    name = name.upper() if assignments["case"] == "upper" else name.lower()
    Proxies.store_of(target.s).shelf.Shelf(target.s, lambda s: s.items({name: lambda e: e.item(lambda i: i.name(name))}))


procedures.register("copy", copy)

<a id="XF-01"></a>
## XF-01 · A transform's anatomy: its name, prototype, parameters and the name its procedure is registered under, defined as data

In [ ]:
Unlabelled = transforms.registered("shelf.Unlabelled")
with transforms.transaction():
    entry = transforms.singleton("Transforms.TheCatalog").transforms.at("Label")
    Label = entry.transform
    assert (str(entry.name), str(Label.procedure)) == ("Label", "label")
    assert [e.pattern for e in Label.prototype] == [Unlabelled]
    (held,) = B.OfPattern.Builder(Unlabelled, False, transforms).parameters  # its parameters: its prototype's
    assert [str(e.name) for e in held.params.parameters] == ["case"]
assert Parametrics.validate(transforms, Unlabelled) == []
assert sorted(transforms.registered("Transforms.Transform").properties) == ["procedure"]  # nothing else
snapshot = Plain.ToPlain(transforms).Snapshot()
transforms.transaction(Transforms.install)  # installed again, defined again: found, nothing written
transforms.transaction(lambda t: Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled))
assert Plain.ToPlain(transforms).Snapshot() == snapshot
assert transforms.registered("Transforms.Transform").ref and certs.registered("Transforms.TransformCertificate").ref

<a id="XF-02"></a>
## XF-02 · Procedures are registered by name, each once; an unknown name is refused before anything runs

In [ ]:
assert "label" in procedures and "ghost" not in procedures
with raises(ValueError, match="a procedure is registered as 'label' already"):
    procedures.register("label", label)
other = Transforms.Procedures()  # a registry is a value: none is global
assert "label" not in other and other.register("label", label) is label
transforms.transaction(lambda t: Transforms.define(t, "Ghost", "ghost", t.shelf.Unlabelled))
data = model("bolt")
with raises(LookupError, match="no procedure is registered as 'ghost'"):
    application("Ghost", {"case": "upper"}, "run", "caller", sync=(data, "shelf.Unlabelled", unlabelled(data, "bolt")))
assert steps("run") == 0 and items(data) == [("bolt", "bolt", None)]

<a id="XF-03"></a>
## XF-03 · A procedure is called with the caller's visitors and the parameter assignments, and that's that: each argument absent is None

In [ ]:
seen = []
procedures.register("inspect", lambda source, sync, target, assignments: seen.append(
    (source, sync, target, dict(assignments), str(source.i.name), source.as_schema().name)))


def frozen(source, sync, target, assignments):
    """Tries to change its parameter assignments."""
    assignments["case"] = "lower"


procedures.register("frozen", frozen)


transforms.transaction(lambda t: Transforms.define(t, "Inspect", "inspect", t.shelf.Unlabelled))
transforms.transaction(lambda t: Transforms.define(t, "Frozen", "frozen", t.shelf.Unlabelled))
data = model("bolt")
frame = unlabelled(data, "bolt")
with transforms.transaction(), data.transaction():
    source = bear(data, "shelf.Unlabelled", frame)
    Transforms.apply(transforms.singleton("Transforms.TheCatalog").transforms.at("Inspect").transform,
                     {"case": "upper"}, procedures, source=source)
    assert seen == [(source, None, None, {"case": "upper"}, "bolt", "shelf.Unlabelled")]  # the very visitor given
    with raises(TypeError, match="does not support item assignment"):  # read-only: what the certificate records is what the procedure was given
        Transforms.apply(transforms.singleton("Transforms.TheCatalog").transforms.at("Frozen").transform,
                         {"case": "upper"}, procedures, source=source)
assert steps("inspect") == 0  # apply writes nothing: certify does

<a id="XF-04"></a>
## XF-04 · The arguments are the caller's: the source read, the sync edited in place and the target built, each a visitor bearing its own pattern frame, in the caller's transactions, which commit or roll back what the procedure wrote

In [ ]:
procedures.register("tag", lambda source, sync, target, assignments: (
    label(source, sync, target, assignments), copy(source, sync, target, assignments)))
transforms.transaction(lambda t: Transforms.define(t, "Tag", "tag", t.shelf.Unlabelled))
source, sync, target = model("bolt", "nut"), model("bolt", "nut"), model()  # the copies are the caller's to make
application("Tag", {"case": "upper"}, "arguments", "caller", source=(source, "shelf.Unlabelled", unlabelled(source, "bolt")),
            sync=(sync, "shelf.Unlabelled", unlabelled(sync, "bolt")), target=(target, "shelf.Whole", whole(target)))
assert items(source) == [("bolt", "bolt", None), ("nut", "nut", None)]  # read
assert items(sync) == [("bolt", "bolt", "BOLT"), ("nut", "nut", None)]  # edited in place
assert items(target) == [("BOLT", "BOLT", None)]  # built

nut = unlabelled(sync, "nut")
with raises(RuntimeError, match="the caller rolls back"):  # neither opens a transaction on the visitors' stores
    with transforms.transaction(), sync.transaction():
        Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform
        given = bear(sync, "shelf.Unlabelled", nut)
        Transforms.apply(Label, {"case": "lower"}, procedures, sync=given)
        Transforms.certify(certs, Label, {"case": "lower"}, "arguments", "caller", sync=given)
        raise RuntimeError("the caller rolls back")
assert items(sync)[1] == ("nut", "nut", None) and steps("arguments") == 2  # rolled back by the caller; certified

<a id="XF-05"></a>
## XF-05 · Parameters: the transform certificate records the parameter assignments, and the history certificate who decided them, as the caller says

In [ ]:
data = model("bolt", "nut")
first = application("Label", {"case": "upper"}, "run", "caller", sync=(data, "shelf.Unlabelled", unlabelled(data, "bolt")))
application("Label", {"case": "lower"}, "run", "policy", sync=(data, "shelf.Unlabelled", unlabelled(data, "nut")))
assert items(data) == [("bolt", "bolt", "BOLT"), ("nut", "nut", "nut")]
with certs.transaction():
    history = certs.singleton("Transforms.TheHistories").histories.at("run").history
    assert [(int(e.index), e.step.assignments.get()) for e in history.steps] == [(0, None), (1, None)]  # a container: not a scalar
    assert [assignments_of(e.step) for e in history.steps] == [{"case": "upper"}, {"case": "lower"}]
    assert [str(d.by) for d in history.decisions] == ["caller", "policy"]  # provenance: the history's, by step
    assert [d.step for d in history.decisions] == [e.step for e in history.steps]
assert Plain.ToPlain(certs)(first.as_schema(), first)["path"]["next"]["entry"]["next"]["entry"]["properties"] == {"index": 0}

<a id="XF-06"></a>
## XF-06 · Certificates are written and linked: the history's sequence, prev and next, the transform and its prototype, a sub-certificate for each argument present holding the pattern instance its visitor bore, and the chain of each pattern instance; all of it data, written and read back as any

In [ ]:
source, target = model("bolt"), model()
bolt, shelf_ = unlabelled(source, "bolt"), whole(target)
for case in ("upper", "lower"):  # the same pattern instances, twice
    application("Copy", {"case": case}, "chain", "caller", source=(source, "shelf.Unlabelled", bolt),
                target=(target, "shelf.Whole", shelf_))
assert items(target) == [("BOLT", "BOLT", None), ("bolt", "bolt", None)]
with transforms.transaction():
    Copy = transforms.singleton("Transforms.TheCatalog").transforms.at("Copy").transform
    copy_ref = References.of(transforms, Copy)
    expected_transform = Plain.ToPlain(transforms)(copy_ref.as_schema(), copy_ref)
pattern = References.of(source, source.registered("shelf.Unlabelled"))
expected_pattern = Plain.ToPlain(source)(pattern.as_schema(), pattern)
instance = References.of(source, {"i": bolt["i"]})  # the frame the visitor bore: its symbols' bindings
expected_frame = Plain.ToPlain(source)(instance.as_schema(), instance)
with certs.transaction():
    history = certs.singleton("Transforms.TheHistories").histories.at("chain").history
    one, two = (e.step for e in history.steps)
    assert [e.after for e in one.next] == [two] and [e.before for e in two.prev] == [one]
    assert list(one.prev) == [] and list(two.next) == []
    assert plain(certs, one.transform) == expected_transform  # the transform, in its own store, by reference
    assert plain(certs, one.prototype) == expected_pattern
    assert [str(e.argument) for e in one.frames] == ["source", "target"]  # whichever are present
    first_source, first_target = (e.frame for e in one.frames)
    second_source, second_target = (e.frame for e in two.frames)
    assert plain(certs, first_source.frame) == expected_frame  # the pattern instance: the frame, by reference
    assert plain(certs, first_target.pattern)["path"]["next"]["entry"]["next"]["entry"]["properties"] == {"name": "Whole"}
    assert list(first_source.preceding) == [] and [e.step for e in second_source.preceding] == [one]
    assert [e.step for e in second_target.preceding] == [one]

log = Plain.ToPlain(certs).Snapshot()  # the certificates are data: written and read back with no code of their own
back = B.Store()
Plain.FromPlain(back).Snapshot(log)
with back.transaction():
    one, two = (e.step for e in back.singleton("Transforms.TheHistories").histories.at("chain").history.steps)
    assert [assignments_of(one), assignments_of(two)] == [{"case": "upper"}, {"case": "lower"}]
    assert [e.before for e in two.prev] == [one] and plain(back, one.prototype) == expected_pattern

<a id="XF-07"></a>
## XF-07 · Verification is the caller's own call, between apply and certify, with the same visitors (the prototype falsified, or a metric decreased); where it fails, the caller does not certify, and no certificate is written

In [ ]:
procedures.register("idle", lambda source, sync, target, assignments: None)
transforms.transaction(lambda t: Transforms.define(t, "Idle", "idle", t.shelf.Unlabelled))


def falsified(sync):
    """The caller's falsification: the sync's item, a frame of shelf.Unlabelled, has a label now."""
    return sync.i.label.get() is not None


data = model("bolt", "nut")
bolt = unlabelled(data, "bolt")
with transforms.transaction(), data.transaction():  # the caller applies, verifies, and certifies only what holds
    catalog = transforms.singleton("Transforms.TheCatalog").transforms
    for name in ("Idle", "Label"):
        transform, sync = catalog.at(name).transform, bear(data, "shelf.Unlabelled", bolt)
        Transforms.apply(transform, {"case": "upper"}, procedures, sync=sync)
        if falsified(sync):  # Idle's fails: not certified
            Transforms.certify(certs, transform, {"case": "upper"}, "verify", "caller", sync=sync)
assert steps("verify") == 1 and items(data)[0] == ("bolt", "bolt", "BOLT")
application("Idle", {"case": "upper"}, "verify", "caller", sync=(data, "shelf.Unlabelled", unlabelled(data, "nut")))  # unverified
assert steps("verify") == 2


def unlabelled_count(sync):
    """How many items on the sync's shelf have no label: a metric."""
    return sum(1 for e in Proxies.store_of(sync.i).singleton("shelf.TheShelf").items if e.item.label.get() is None)


data = model("nut")
nut = unlabelled(data, "nut")
with transforms.transaction(), data.transaction():  # a metric: the caller computes it before, and compares after
    catalog = transforms.singleton("Transforms.TheCatalog").transforms
    sync = bear(data, "shelf.Unlabelled", nut)
    for name in ("Idle", "Label"):
        before = unlabelled_count(sync)
        Transforms.apply(catalog.at(name).transform, {"case": "lower"}, procedures, sync=sync)
        if unlabelled_count(sync) < before:  # Idle's has not decreased: not certified
            Transforms.certify(certs, catalog.at(name).transform, {"case": "lower"}, "metric", "caller", sync=sync)
assert steps("metric") == 1 and items(data) == [("nut", "nut", "nut")]
with certs.transaction():  # what was certified is Label's application
    histories = certs.singleton("Transforms.TheHistories").histories
    assert [plain(certs, histories.at(h).history.steps.at(0).step.transform)["path"]["next"]["entry"]["properties"]
            for h in ("verify", "metric")] == [{"name": "Label"}] * 2


<a id="XF-08"></a>
## XF-08 · What an application refuses: a transform with no prototype, and a procedure that raises writes nothing

In [ ]:
transforms.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"catalog": t.Transforms.Catalog, "transform": t.Transforms.Transform}).spec([
        lambda _: _.catalog.transforms({"Bare": lambda e: e.transform(_.transform)}),
        lambda _: _.transform.procedure("label")])]))]))
data = model("bolt")
with raises(ValueError, match="transform 'Bare' has no prototype"):
    application("Bare", {"case": "upper"}, "refused", "caller", sync=(data, "shelf.Unlabelled", unlabelled(data, "bolt")))


def broken(source, sync, target, assignments):
    label(source, sync, target, assignments)
    raise RuntimeError("the procedure failed")


procedures.register("broken", broken)
transforms.transaction(lambda t: Transforms.define(t, "Broken", "broken", t.shelf.Unlabelled))
with raises(RuntimeError, match="the procedure failed"):
    application("Broken", {"case": "upper"}, "refused", "caller", sync=(data, "shelf.Unlabelled", unlabelled(data, "bolt")))
assert items(data) == [("bolt", "bolt", None)] and steps("refused") == 0  # the caller's transaction rolled back